In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import pandas as pd
import numpy as np
import torch
from transformers import pipeline , AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from sentence_transformers import CrossEncoder
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
!pip install transformers datasets -q
import wandb
from kaggle_secrets import UserSecretsClient




In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("GPU:", torch.cuda.is_available())

def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

OPTIONS  = ['A', 'B', 'C', 'D', 'E']
LABEL2ID = {opt: i for i, opt in enumerate(OPTIONS)}
ID2LABEL = {i: opt for opt, i in LABEL2ID.items()}

In [ ]:
# PREPROCESSING

def preprocess(examples):
    first_flat  = []
    second_flat = []

    for i in range(len(examples['prompt'])):
        for opt in OPTIONS:
            first_flat.append(str(examples['prompt'][i]))
            second_flat.append(str(examples[opt][i]))

    tokenized = tokenizer(
        first_flat,
        second_flat,
        truncation=True,
        max_length=128,
        padding='max_length'
    )

    result = {
        k: [v[i:i+5] for i in range(0, len(v), 5)]
        for k, v in tokenized.items()
    }

    if 'answer' in examples:
        result['labels'] = [LABEL2ID[ans] for ans in examples['answer']]

    return result

# Train/val split
train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test.reset_index(drop=True))

# Tokenize
cols_to_remove_train = train_dataset.column_names
cols_to_remove_test  = test_dataset.column_names

train_tok = train_dataset.map(preprocess, batched=True, remove_columns=cols_to_remove_train)
val_tok   = val_dataset.map(preprocess,   batched=True, remove_columns=cols_to_remove_train)
test_tok  = test_dataset.map(preprocess,  batched=True, remove_columns=cols_to_remove_test)

print("Tokenization done!")
print(f"Sample keys: {list(train_tok[0].keys())}")

In [ ]:
# Model

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
model = model.float()

print(next(model.parameters()).dtype)

In [ ]:
# MAP@3 METRIC

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = []
    for logit_row, label in zip(logits, labels):
        ranked  = np.argsort(logit_row)[::-1]
        top3    = [ID2LABEL[i] for i in ranked[:3]]
        correct = ID2LABEL[label]
        for k, pred in enumerate(top3):
            if pred == correct:
                scores.append(1.0 / (k + 1))
                break
        else:
            scores.append(0.0)
    return {"MAP@3": round(sum(scores)/len(scores), 4)}

In [ ]:
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

In [ ]:
# TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta-mcq",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="MAP@3",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="wandb",
    run_name="deberta-v3-small-finetuned",
)

In [ ]:
# TRAIN

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    processing_class=tokenizer, 
    compute_metrics = compute_metrics,
)

print("Starting training... (takes ~40 mins on T4)")
print("Watch MAP@3 improve each epoch!\n")
trainer.train()

In [ ]:
# CHECK BEST SCORE AFTER TRAINING

print("\nTraining complete!")
print("Best MAP@3 on validation:", 
      trainer.state.best_metric)

# Evaluate one more time on val set
results = trainer.evaluate()
print("Final val results:", results)

In [ ]:
# GENERATE TEST SUBMISSION

print("Generating predictions on test set...")
raw_preds = trainer.predict(test_tok)
logits    = raw_preds.predictions  # shape: (500, 5)

predictions = []
for i, (logit_row, (_, row)) in enumerate(zip(logits, test.iterrows())):
    ranked = np.argsort(logit_row)[::-1]
    top3   = ' '.join([ID2LABEL[j] for j in ranked[:3]])
    predictions.append({'id': row['id'], 'prediction': top3})

submission = pd.DataFrame(predictions)
submission.to_csv('/kaggle/working/submission.csv', index=False)


In [ ]:
'''
This submission uses a fine-tuned DeBERTa model for solving the MCQ task.
Before this,  I directly used pre-trained models, 
this approach trains the DeBERTa model on the Smart MCQ training dataset before making predictions.
For each question, the prompt and all five answer options are given as input to the model. 
During training, the model learns the relationship between the question and the correct answer by using the provided training labels. 
After training, the model predicts a score for each of the five answer options. 
These scores are then ranked, and the top three answer options are selected for the final submission.
The model was trained for multiple epochs using the Hugging Face Trainer, and the best-performing model was selected based on the MAP@3 evaluation metric.
By using this method, the public leaderboard score obtained was 0.73607.

**Conclusion:**

This approach achieved a significant improvement over all my previous submissions. 
The score increased from 0.50623 to 0.73607.
The improvement is because the model was fine-tuned on the training dataset instead of relying only on pre-trained knowledge. 
During training, the model learned the patterns between the questions and their correct answers, allowing it to make more accurate predictions.
This experiment shows that fine-tuning a transformer model is much more effective for this task than using similarity-based methods or zero-shot classification. 
Further improvements may be possible by tuning the training parameters, using a larger model, or combining multiple fine-tuned models.

'''